# How the secondary and safety outcomes are derived

Companion to `primary_outcome.ipynb`, which builds the cohort, the propensity score `e(X)` and the
overlap weights `w`. This notebook reuses those steps unchanged and shows what happens **after** them
for the seven *binary* outcomes `[§5]`:

| Family | Outcome | Good or bad event? |
|---|---|---|
| Secondary | mRS 0–2 · mRS 0–1 · TICI 2b–3 | good |
| Safety | symptomatic ICH · parenchymal haematoma type 2 · death · mRS 5–6 | bad |

Each outcome gets up to three numbers `[§8]`:

```
weighted risk difference  RD   ──►  the basic contrast
weighted marginal odds ratio    ──►  the same contrast on the odds scale
model-assisted augmented RD τ   ──►  RD with a correction from an outcome model (when estimable)
```

All three are computed in the same overlap population as the primary. Section references `[§n]`
point to `statistical_analysis_plan.md`.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd().parent))     # the analysis modules live one directory up

import config as C
import data, derive, eligibility, cohort, propensity, outcome, bootstrap, model, sensitivity, standardise, policy

pd.set_option("display.precision", 3)
plt.rcParams["figure.dpi"] = 110

ARM = C.TREATMENT_LABELS        # {0: "EVT alone", 1: "bridging"}
COLOURS = {0: "#7f8c8d", 1: "#2980b9"}

## 1. Cohort, propensity score and weights — as for the primary

In [ ]:
df, audit = data.load(data.WORKBOOK)
df = derive.derive(df, audit)
df = eligibility.classify(df, audit)
df = cohort.build(df, audit)
ps = propensity.fit(df, audit)

print(f"cohort {len(df)}, with a propensity score {ps.in_model.sum()}")

## 2. Where the binary outcomes come from

The three mRS dichotomies and mRS 5–6 are **cut from the ordinal 90-day mRS**, never read from
separate workbook columns `[§5]`. So they inherit exactly its missingness, and `RD` for mRS 0–2 equals
the primary notebook's `RD_2`. TICI, sICH and PH2 are recorded directly.

In [ ]:
def label(key):
    return C.OUTCOMES[key].label


events = pd.DataFrame({
    "family": {label(k): C.OUTCOMES[k].family for k in C.BINARY_OUTCOMES},
    "events": {label(k): int(df[k].sum()) for k in C.BINARY_OUTCOMES},
    "non-events": {label(k): int((df[k] == 0).sum()) for k in C.BINARY_OUTCOMES},
    "missing": {label(k): int(df[k].isna().sum()) for k in C.BINARY_OUTCOMES},
})
events["minority cell"] = events[["events", "non-events"]].min(axis=1)
events

The **minority cell** — the smaller of events and non-events — decides how much modelling an outcome can
carry (§4 below). The rule uses `min(events, non-events)` rather than events alone because a model is
starved by whichever cell is small: TICI 2b–3 has many events but few *non*-events.

## 3. The two basic estimates: weighted RD and weighted odds ratio

With the overlap weights, each arm's outcome rate is a weighted mean. Then, for every outcome:

```
RD = P_w(event | bridging) − P_w(event | EVT alone)
OR = odds_w(event | bridging) / odds_w(event | EVT alone)
```

**Sign convention:** `RD > 0` always means *more events under bridging*. For a good outcome (mRS 0–2)
that favours bridging; for a bad one (death) it does not. The sign belongs to the contrast, not the
outcome, so every row of a table means the same thing.

In [ ]:
def weighted_rates(key):
    # Per-arm weighted event rate on the outcome's own population: weighted AND outcome present.
    m = ps.in_model & df[key].notna()
    y, a, w = df.loc[m, key].to_numpy(float), df.loc[m, C.TREATMENT].to_numpy(float), ps.w[m].to_numpy(float)
    return outcome.weighted_proportion(y, a, w), outcome.weighted_rd(y, a, w), int(m.sum())


rows = []
for key in C.BINARY_OUTCOMES:
    share, rd, n = weighted_rates(key)
    rows.append((label(key), n, share[0], share[1], rd))
basic = pd.DataFrame(rows, columns=["outcome", "n", f"rate, {ARM[0]}", f"rate, {ARM[1]}", "RD"]).set_index("outcome")
basic

The odds ratio has one wrinkle: when an arm has a weighted rate of exactly 0 or 1 it is undefined. The
code then adds a small continuity correction to all four cells and **flags** the row, so a corrected
number is never mistaken for a raw one. That flag appears as `or_corrected` in §5.

## 4. The augmented estimate: borrowing from an outcome model

Weighting alone removes confounding only as well as the propensity model allows. A second model —
`m_a(X)`, the probability of the event given treatment and the same confounders — can mop up residual
imbalance. The correction is `[§8]`:

```
τ = Σ₁ w(Y − m₁)/Σ₁ w  −  Σ₀ w(Y − m₀)/Σ₀ w  +  Σ h(m₁ − m₀)/Σ h        with  h = e(1 − e)
```

Read it as: *(weighted residuals in each arm)* + *(model-predicted effect, averaged over the overlap
population)*. If `m_a(X)` is a constant, `τ` collapses exactly to the plain `RD`, which is what makes
the two comparable.

Two honest limits, stated in the plan:

- **Not "doubly robust".** The overlap population itself is defined by `e(X)`. If `e(X)` is wrong the
  *target* is wrong, however good `m_a(X)` is. Hence the label *model-assisted*.
- **Needs events.** A model with more parameters than minority-cell observations is noise. Rule:
  minority cell `< 10` → no augmentation (sICH, PH2). TICI 2b–3 uses a prespecified *reduced* model
  (treatment + centre + atrial fibrillation) instead of the full confounder set.

In [ ]:
sec = outcome.secondary(df, ps, audit)

paths = pd.DataFrame({
    "minority cell": {label(k): e.minority for k, e in sec.estimates.items()},
    "augmentation": {label(k): e.augmented_path for k, e in sec.estimates.items()},
    "m_a(X) covariates": {label(k): ", ".join(e.covariates) if e.covariates else "—" for k, e in sec.estimates.items()},
})
paths

The three point estimates side by side. `RD` and `τ` should be close; a large gap says something about the outcome model, not about which one is right.

In [ ]:
point = pd.DataFrame({
    "family": {label(k): e.family for k, e in sec.estimates.items()},
    "RD": {label(k): e.rd for k, e in sec.estimates.items()},
    "τ (augmented)": {label(k): e.augmented for k, e in sec.estimates.items()},
    "odds ratio": {label(k): e.odds_ratio for k, e in sec.estimates.items()},
    "or_corrected": {label(k): e.or_corrected for k, e in sec.estimates.items()},
})
point

## 5. Uncertainty: the same bootstrap as the primary

2000 centre-stratified resamples; in each, the propensity model, the weights, every outcome model and
every estimate are refitted from scratch `[§10]`. Two rules specific to the binary outcomes:

- **The augmentation path is frozen.** Whether an outcome is augmented is decided *once*, on the full
  data, and reused in every replicate. PH2 sits near the 10-event threshold and would otherwise flip
  between estimators from replicate to replicate, making the interval a mixture of two estimators.
- **p-values come from the RD draws, never the OR draws.** The OR is undefined in replicates where a
  rate hits 0 or 1; taking p from the surviving OR draws would select on the outcome. `RD` exists in
  every replicate, and `RD = 0` is the same null as `OR = 1`.

This cell takes about three minutes.

In [ ]:
est = outcome.primary(df, ps, audit)                   # bootstrap.run needs it for the shared replicate loop
boot = bootstrap.run(df, ps, est, sec, audit)

def ci(key):
    iv = boot.intervals[key]
    return iv.lo, iv.hi, iv.p, iv.n_draws


rows = []
for k, e in sec.estimates.items():
    rd_lo, rd_hi, p, n = ci(f"{k}.rd")
    or_lo, or_hi, _, n_or = ci(f"{k}.odds_ratio")
    tau = (np.nan, np.nan) if e.augmented is None else ci(f"{k}.augmented")[:2]
    rows.append((label(k), e.rd, rd_lo, rd_hi, p, n, e.odds_ratio, or_lo, or_hi, n_or, e.augmented, *tau))

results = pd.DataFrame(rows, columns=["outcome", "RD", "lo", "hi", "p", "draws",
                                      "OR", "lo ", "hi ", "OR draws", "τ", "lo  ", "hi  "]).set_index("outcome")
results

**`OR draws` < 2000** is the undefined-odds problem made visible: those replicates were dropped and counted, not patched.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
names = [label(k) for k in C.BINARY_OUTCOMES][::-1]
ypos = np.arange(len(names))

for ax, (col, lo, hi, null, xlabel) in zip(axes, [("RD", "lo", "hi", 0.0, "risk difference (bridging − EVT alone)"),
                                                  ("OR", "lo ", "hi ", 1.0, "weighted marginal odds ratio")]):
    r = results.loc[names]
    ax.errorbar(r[col], ypos, xerr=[r[col] - r[lo], r[hi] - r[col]], fmt="o", color="#2c3e50", capsize=3)
    ax.axvline(null, color="red", ls="--")
    ax.set_yticks(ypos, names); ax.set_xlabel(xlabel)
    if null == 1.0:
        ax.set_xscale("log")
plt.tight_layout(); plt.show()

## 6. Multiplicity: seven outcomes, two families

Seven tests raise the chance that one falls below 0.05 by luck. The plan applies Benjamini–Hochberg
false-discovery-rate correction **within** each family — three secondary, four safety `[§13]`. The
primary p-value is left uncorrected because it is the one prespecified test.

Safety outcomes rest on very few events and are labelled *descriptive*: the intervals are reported,
but no conclusion is drawn from a p-value.

In [ ]:
mult = sensitivity.multiplicity(sec, boot, audit)

rows = []
for fam in mult.families.values():
    for k in fam.raw:
        rows.append((fam.family, label(k), fam.raw[k], fam.adjusted[k], "descriptive only" if fam.descriptive_only else ""))
pd.DataFrame(rows, columns=["family", "outcome", "p raw", "p BH-adjusted", "note"]).set_index(["family", "outcome"])

## 7. Reading the results

- **Three numbers, one contrast.** `RD` is the reportable estimate; `OR` is the same thing on another
  scale; `τ` is a model-assisted refinement read *beside* `RD`, never instead of it.
- **Signs.** Positive = more events with bridging, for good and bad outcomes alike.
- **Missing `τ`** means the outcome was too rare to model, by a rule fixed before any data were seen.
- **Descriptive** rows (safety) carry intervals but no inferential claim.
- **Population.** Same overlap population as the primary: IVT-eligible patients at both-arm centres
  who could realistically have received either treatment.